## 1. Imports y carga inicial

In [0]:
from pyspark.sql import functions as f

BASE = "/Volumes/mine4213/proyecto/data"
CSV_DIR = f"{BASE}/csv"

AIS_SCHEMA = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG float,
    COG float,
    Heading float,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType smallint,
    Status smallint,
    Length float,
    Width float,
    Draft float,
    Cargo string,
    TransceiverClass string
"""

ais = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(AIS_SCHEMA)
    .csv(f"{CSV_DIR}/AIS_2023_06_*.csv")
    .withColumn("fecha", f.to_date("BaseDateTime"))
    .withColumn(
        "archivo_origen",
        f.regexp_extract(
            f.col("_metadata.file_name"),
            r"(AIS_2023_06_\d{2}\.csv)",
            1
        )
    )
    .withColumn(
        "fecha_archivo",
        f.to_date(
            f.regexp_extract(
                f.col("_metadata.file_name"),
                r"AIS_(\d{4}_\d{2}_\d{2})\.csv",
                1
            ),
            "yyyy_MM_dd"
        )
    )
)

ais.printSchema()

## 2. Posiciones y buques unicos por dia

In [0]:
perfil_diario = (
    ais
    .groupBy("fecha")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .orderBy("fecha")
)

display(perfil_diario)

Databricks visualization. Run in Databricks to view.

### Total semanal

In [0]:
resumen_general = (
    ais
    .agg(
        f.count("*").alias("total_posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos_semana"),
        f.min("BaseDateTime").alias("primer_timestamp"),
        f.max("BaseDateTime").alias("ultimo_timestamp")
    )
)

display(resumen_general)

## 3. Completitud

In [0]:
columnas_originales = [
    "MMSI",
    "BaseDateTime",
    "LAT",
    "LON",
    "SOG",
    "COG",
    "Heading",
    "VesselName",
    "IMO",
    "CallSign",
    "VesselType",
    "Status",
    "Length",
    "Width",
    "Draft",
    "Cargo",
    "TransceiverClass"
]

string_cols = {
    field.name
    for field in ais.schema.fields
    if field.dataType.simpleString() == "string"
}

missing_exprs = []

for col_name in columnas_originales:

    if col_name in string_cols:
        missing_condition = (
            f.col(col_name).isNull()
            | (f.trim(f.col(col_name)) == "")
        )
    else:
        missing_condition = f.col(col_name).isNull()

    missing_exprs.append(
        f.sum(
            f.when(missing_condition, 1).otherwise(0)
        ).alias(col_name)
    )

missing_row = (
    ais
    .agg(
        f.count("*").alias("total_filas"),
        *missing_exprs
    )
    .first()
)

total_filas = missing_row["total_filas"]

perfil_completitud = spark.createDataFrame(
    [
        (
            col_name,
            int(missing_row[col_name]),
            round(
                100 * missing_row[col_name] / total_filas,
                4
            )
        )
        for col_name in columnas_originales
    ],
    [
        "columna",
        "valores_faltantes",
        "porcentaje_faltante"
    ]
)

display(
    perfil_completitud
    .orderBy(f.desc("porcentaje_faltante"))
)

## 4. Distribucion por tipo de buqeu

In [0]:
distribucion_tipo = (
    ais
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .withColumn(
        "porcentaje_posiciones",
        f.round(
            100
            * f.col("posiciones")
            / f.lit(total_filas),
            4
        )
    )
    .orderBy(f.desc("posiciones"))
)

display(distribucion_tipo)

In [0]:
display(
    distribucion_tipo
    .filter(f.col("VesselType").isNull())
)

In [0]:
display(distribucion_tipo.limit(20))

## 5. Distribucion por tama;o

In [0]:
dimensiones_validas = (
    ais
    .select(
        f.when(f.col("Length") > 0, f.col("Length")).alias("Length"),
        f.when(f.col("Width") > 0, f.col("Width")).alias("Width"),
        f.when(f.col("Draft") > 0, f.col("Draft")).alias("Draft")
    )
)

display(
    dimensiones_validas.summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "90%",
        "95%",
        "99%",
        "max"
    )
)

### Revision dimensiones extremas

In [0]:
display(
    ais
    .select(
        "MMSI",
        "VesselName",
        "VesselType",
        "Length",
        "Width",
        "Draft"
    )
    .orderBy(
        f.desc("Length")
    )
    .limit(50)
)

In [0]:
display(
    ais
    .select(
        "MMSI",
        "VesselName",
        "VesselType",
        "Length",
        "Width",
        "Draft"
    )
    .orderBy(
        f.desc("Width")
    )
    .limit(50)
)

In [0]:
ais_tamano = (
    ais
    .withColumn(
        "categoria_tamano",
        f.when(
            f.col("Length").isNull() | (f.col("Length") <= 0),
            "Sin longitud utilizable"
        )
        .when(f.col("Length") < 25, "< 25 m")
        .when(f.col("Length") < 50, "25 - 49.9 m")
        .when(f.col("Length") < 100, "50 - 99.9 m")
        .when(f.col("Length") < 200, "100 - 199.9 m")
        .otherwise(">= 200 m")
    )
)

distribucion_tamano = (
    ais_tamano
    .groupBy("categoria_tamano")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .withColumn(
        "porcentaje_posiciones",
        f.round(
            100
            * f.col("posiciones")
            / f.lit(total_filas),
            4
        )
    )
    .orderBy(f.desc("posiciones"))
)

display(distribucion_tamano)

In [0]:
tipo_tamano = (
    ais_tamano
    .groupBy(
        "VesselType",
        "categoria_tamano"
    )
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .orderBy(f.desc("posiciones"))
)

display(tipo_tamano.limit(50))

## 6. Reglas de caldiad

In [0]:
inicio_semana = f.lit("2023-06-01 00:00:00").cast("timestamp")
fin_semana = f.lit("2023-06-08 00:00:00").cast("timestamp")

sog_no_disponible = (
    f.abs(f.col("SOG") - f.lit(102.3)) < 0.001
)

checks = [
    (
        "coordenadas_nulas",
        f.col("LAT").isNull() | f.col("LON").isNull(),
        "LAT o LON no disponibles"
    ),
    (
        "lat_fuera_rango",
        (f.col("LAT") < -90) | (f.col("LAT") > 90),
        "Latitud fuera de [-90, 90]"
    ),
    (
        "lon_fuera_rango",
        (f.col("LON") < -180) | (f.col("LON") > 180),
        "Longitud fuera de [-180, 180]"
    ),

    (
        "sog_nulo",
        f.col("SOG").isNull(),
        "Velocidad no informada"
    ),
    (
        "sog_no_disponible_102_3",
        sog_no_disponible,
        "102.3 corresponde al valor AIS de SOG no disponible"
    ),
    (
        "sog_fuera_codificacion_ais",
        (f.col("SOG") < 0)
        | (
            (f.col("SOG") > 102.3)
            & (~sog_no_disponible)
        ),
        "Valor fuera del rango esperado para SOG AIS"
    ),
    (
        "sog_mayor_60_sospechosa",
        (f.col("SOG") > 60) & (f.col("SOG") <= 102.2),
        "Velocidad muy alta: revisar, no eliminar automáticamente"
    ),

    (
        "cog_no_disponible_360",
        f.abs(f.col("COG") - f.lit(360.0)) < 0.001,
        "COG=360 significa no disponible"
    ),
    (
        "cog_fuera_rango",
        (f.col("COG") < 0) | (f.col("COG") > 360),
        "COG fuera del rango AIS"
    ),

    (
        "heading_no_disponible_511",
        f.abs(f.col("Heading") - f.lit(511.0)) < 0.001,
        "Heading=511 significa no disponible"
    ),
    (
        "heading_fuera_rango",
        (f.col("Heading") < 0)
        | (
            (f.col("Heading") > 359)
            & (f.abs(f.col("Heading") - f.lit(511.0)) >= 0.001)
        ),
        "Heading distinto de 0-359 y no es el sentinel 511"
    ),

    (
        "mmsi_nulo_o_vacio",
        f.col("MMSI").isNull()
        | (f.trim(f.col("MMSI")) == ""),
        "MMSI ausente"
    ),
    (
        "mmsi_formato_invalido",
        f.col("MMSI").isNotNull()
        & (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")),
        "MMSI que no contiene exactamente 9 dígitos"
    ),

    (
        "timestamp_nulo",
        f.col("BaseDateTime").isNull(),
        "Timestamp ausente"
    ),
    (
        "timestamp_fuera_semana",
        (f.col("BaseDateTime") < inicio_semana)
        | (f.col("BaseDateTime") >= fin_semana),
        "Timestamp fuera del corpus 1-7 junio"
    ),
    (
        "fecha_no_coincide_archivo",
        f.col("fecha").isNotNull()
        & f.col("fecha_archivo").isNotNull()
        & (f.col("fecha") != f.col("fecha_archivo")),
        "El timestamp no corresponde al día indicado por el archivo"
    ),

    (
        "length_negativo",
        f.col("Length") < 0,
        "Longitud físicamente inválida"
    ),
    (
        "width_negativo",
        f.col("Width") < 0,
        "Ancho físicamente inválido"
    ),
    (
        "draft_negativo",
        f.col("Draft") < 0,
        "Calado físicamente inválido"
    ),

    (
        "length_cero",
        f.col("Length") == 0,
        "Longitud cero; posiblemente no disponible"
    ),
    (
        "width_cero",
        f.col("Width") == 0,
        "Ancho cero; posiblemente no disponible"
    ),
    (
        "draft_cero",
        f.col("Draft") == 0,
        "Calado cero; posiblemente no disponible"
    ),

    (
        "vessel_type_nulo",
        f.col("VesselType").isNull(),
        "Tipo de buque no informado"
    )
]

In [0]:
quality_exprs = []

for i, (_, condition, _) in enumerate(checks):
    quality_exprs.append(
        f.sum(
            f.when(condition, 1).otherwise(0)
        ).alias(f"q_{i}")
    )

quality_row = (
    ais
    .agg(
        f.count("*").alias("total_filas"),
        *quality_exprs
    )
    .first()
)

total_filas = quality_row["total_filas"]

In [0]:
quality_data = []

for i, (nombre, _, descripcion) in enumerate(checks):

    cantidad = int(quality_row[f"q_{i}"])

    porcentaje = round(
        100 * cantidad / total_filas,
        6
    )

    quality_data.append(
        (
            nombre,
            cantidad,
            porcentaje,
            descripcion
        )
    )

diagnostico_calidad = spark.createDataFrame(
    quality_data,
    [
        "regla",
        "filas_afectadas",
        "porcentaje",
        "interpretacion"
    ]
)

display(
    diagnostico_calidad
    .orderBy(f.desc("filas_afectadas"))
)

### Headings anomalos

In [0]:
heading_anomalos = (
    ais
    .filter(
        (f.col("Heading") < 0)
        | (
            (f.col("Heading") > 359)
            & (f.abs(f.col("Heading") - 511.0) >= 0.001)
        )
    )
    .groupBy("Heading")
    .agg(
        f.count("*").alias("apariciones")
    )
    .orderBy(
        f.desc("apariciones")
    )
)

display(heading_anomalos)

In [0]:
cog_anomalos = (
    ais
    .filter(
        (f.col("COG") < 0)
        | (f.col("COG") > 360)
    )
    .groupBy("COG")
    .agg(
        f.count("*").alias("apariciones")
    )
    .orderBy(
        f.desc("apariciones")
    )
)

display(cog_anomalos)

## 7. Velocidades altas

In [0]:
display(
    ais
    .filter(
        (f.col("SOG") > 60)
        & (f.col("SOG") <= 102.2)
    )
    .select(
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON",
        "SOG",
        "VesselType",
        "VesselName"
    )
    .orderBy(f.desc("SOG"))
    .limit(100)
)

In [0]:
display(
    ais
    .select("SOG")
    .filter(f.col("SOG").isNotNull())
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "50%",
        "90%",
        "95%",
        "99%",
        "max"
    )
)

## 8. Duplicados
Duplciado = mismo MMSI + BaseDateTime

In [0]:
duplicados_timestamp = (
    ais
    .filter(
        f.col("MMSI").isNotNull()
        & f.col("BaseDateTime").isNotNull()
    )
    .groupBy(
        "MMSI",
        "BaseDateTime"
    )
    .agg(
        f.count("*").alias("numero_registros"),
        f.countDistinct(
            f.struct("LAT", "LON")
        ).alias("posiciones_distintas")
    )
    .filter(
        f.col("numero_registros") > 1
    )
)

In [0]:
resumen_duplicados = (
    duplicados_timestamp
    .agg(
        f.count("*").alias(
            "claves_mmsi_timestamp_duplicadas"
        ),

        f.sum("numero_registros").alias(
            "filas_en_claves_duplicadas"
        ),

        f.sum(
            f.col("numero_registros") - 1
        ).alias(
            "filas_excedentes"
        ),

        f.sum(
            f.when(
                f.col("posiciones_distintas") == 1,
                1
            ).otherwise(0)
        ).alias(
            "claves_con_misma_posicion"
        ),

        f.sum(
            f.when(
                f.col("posiciones_distintas") > 1,
                1
            ).otherwise(0)
        ).alias(
            "claves_con_posiciones_conflictivas"
        )
    )
)

display(resumen_duplicados)

In [0]:
display(
    duplicados_timestamp
    .orderBy(
        f.desc("numero_registros")
    )
    .limit(100)
)

In [0]:
display(
    duplicados_timestamp
    .filter(
        f.col("posiciones_distintas") > 1
    )
    .orderBy(
        f.desc("numero_registros")
    )
    .limit(100)
)

### Duplicados por contenido completo

In [0]:
ais_hash = (
    ais
    .withColumn(
        "_row_hash",
        f.xxhash64(
            *[f.col(c) for c in columnas_originales]
        )
    )
)

duplicados_fila = (
    ais_hash
    .groupBy("_row_hash")
    .agg(
        f.count("*").alias("numero_registros")
    )
    .filter(
        f.col("numero_registros") > 1
    )
)

In [0]:
resumen_duplicados_fila = (
    duplicados_fila
    .agg(
        f.count("*").alias(
            "grupos_repetidos"
        ),
        f.sum("numero_registros").alias(
            "filas_en_grupos_repetidos"
        ),
        f.sum(
            f.col("numero_registros") - 1
        ).alias(
            "filas_excedentes"
        )
    )
)

display(resumen_duplicados_fila)

## 9. MMSI anomalso

In [0]:
mmsi_anomalos = (
    ais
    .filter(
        f.col("MMSI").isNull()
        | (f.trim(f.col("MMSI")) == "")
        | (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$"))
    )
    .groupBy("MMSI")
    .agg(
        f.count("*").alias("posiciones")
    )
    .orderBy(
        f.desc("posiciones")
    )
)

display(mmsi_anomalos)

## 10. Coordenadas anomalas

In [0]:
display(
    ais
    .filter(
        (f.col("LAT") < -90)
        | (f.col("LAT") > 90)
        | (f.col("LON") < -180)
        | (f.col("LON") > 180)
    )
    .groupBy(
        "LAT",
        "LON"
    )
    .agg(
        f.count("*").alias("apariciones")
    )
    .orderBy(
        f.desc("apariciones")
    )
)

## 11. Problemas de calidad por dia

In [0]:
calidad_por_dia = (
    ais
    .groupBy("fecha")
    .agg(
        f.count("*").alias("posiciones"),

        f.sum(
            f.when(
                (f.col("LAT") < -90)
                | (f.col("LAT") > 90)
                | (f.col("LON") < -180)
                | (f.col("LON") > 180),
                1
            ).otherwise(0)
        ).alias("coordenadas_fuera_rango"),

        f.sum(
            f.when(
                f.abs(f.col("SOG") - 102.3) < 0.001,
                1
            ).otherwise(0)
        ).alias("sog_no_disponible"),

        f.sum(
            f.when(
                f.col("MMSI").isNull()
                | (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")),
                1
            ).otherwise(0)
        ).alias("mmsi_anomalo")
    )
    .orderBy("fecha")
)

display(calidad_por_dia)